# 11 · Training: the attack → detect → patch loops

Each loop starts from the released R0 of its seed and runs three rounds. In round *k* a fresh attacker is trained
against R<sub>k-1</sub>, the attacked audio that scores higher *and* is measurably worse on the detector panel becomes a
preference pair, and R<sub>k-1</sub> is fine-tuned on all pairs so far to give R<sub>k</sub>.

* `hnp_s0..2` — the main loop (EnCodec attack space), about 1.3 h each.
* `abl_*_s0` — ablations: detection threshold τ (0.1, 0.5), ranking margin (0.1, 1.0), head-only patching, and
  using the raw input instead of the codec reconstruction as the preferred side.
* `ms_s0..2` — multi-space loop (EnCodec + DAC + waveform PGD attacks each round), about 2.3 h each.

The main and ablation loops were run before multi-space support existed; the stored configs have no
`loop.attack_spaces`, so retraining adds the single EnCodec space (identical behaviour except the per-round seed
formula, `seed*1000+k` then vs `seed*1000+10k` now).

In [1]:
import os, sys
from pathlib import Path

# Works when started from the repo root or from notebooks/; needs the project environment
# (`source scripts/env.sh`, kernel "hackpatch" — see 00_setup.ipynb).
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from IPython.display import Markdown, display
from hackpatch import repro

pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_rows", 200)

In [2]:
# Parameters
# LAUNCH = False only shows what would run (the job script of the first key); True starts the training.
LAUNCH = os.environ.get("NB_LAUNCH", "0") == "1"
# RUN = "slurm": one GPU job per run (sbatch); "local": in this kernel (needs a GPU, runs sequentially).
RUN = os.environ.get("NB_RUN", "slurm")

In [3]:
reg = repro.registry()
KEYS = [k for k, v in reg.items() if v["kind"] in ['hnp', 'ablation', 'multispace']]
print(len(KEYS), "runs:", ", ".join(KEYS))

12 runs: hnp_s0, hnp_s1, hnp_s2, abl_tau0.1_s0, abl_margin0.1_s0, abl_headonly_s0, abl_tau0.5_s0, abl_margin1.0_s0, abl_prefinput_s0, ms_s0, ms_s1, ms_s2


In [4]:
# Resolved configuration of the first run (checkpoint paths already point at the local released copies).
from omegaconf import OmegaConf
print(OmegaConf.to_yaml(repro.load_config(KEYS[0])))

config_name: hack_loop
task: hack_loop
phase: main
seed: 0
run_name: hack_loop_s0__2026-09-15_16-10-10
init_ckpt: /fs/scratch/PAS2301/alialavi/cache/audio_assessment_hackability_release/checkpoints/r0_s0.checkpoints.best.pt
model:
  pretrained: facebook/wav2vec2-base
  freeze_feature_encoder: true
  freeze_encoder: false
data:
  corpus: bvcc
  max_seconds: 6.0
  num_workers: 8
loop:
  rounds: 3
  pair_preferred: recon
  val_attack_utts: 400
  max_train_utts: null
attack:
  codec:
    name: encodec_24khz
    quantize: false
    encodec_bandwidth: 24.0
  policy:
    hidden: 256
    layers: 4
    kernel: 5
    init_log_std: -1.0
  action_hop: 0
  budget_mode: sphere
  eps: 0.5
  budget_penalty: 0.0
  group_size: 8
  batch_size: 8
  crop_seconds: 3.0
  steps: 2000
  lr: 0.001
  ppo_epochs: 2
  clip: 0.2
  adv_eps: 0.0001
  grad_clip: 1.0
  log_every: 10
  apply:
    n_samples: 3
    batch_size: 16
    chunk_utts: 512
    max_seconds: 20.0
detect:
  tau: 0.3
  measures:
  - estoi
  - sisdr


In [5]:
if LAUNCH:
    repro.run_keys(KEYS, "retrain", RUN)
else:
    print("LAUNCH=False: nothing started. Job script for", KEYS[0], "->")
    repro.submit(KEYS[0], "retrain", dry_run=True)

LAUNCH=False: nothing started. Job script for hnp_s0 ->
#!/bin/bash
#SBATCH --account=PAS2966
#SBATCH --partition=nextgen
#SBATCH --job-name=retrain_hnp_s0
#SBATCH --output=/fs/scratch/PAS2301/alialavi/projects/audio_assessment_hackability/retrain/_logs/%x_%j.out
#SBATCH --time=02:00:00
#SBATCH --nodes=1
#SBATCH --ntasks-per-node=1
#SBATCH --cpus-per-task=12
#SBATCH --mem=64G
#SBATCH --gpus-per-node=1

set -euo pipefail
cd /users/PAS2301/alialavi/projects/audio_assessment_hackability
source scripts/env.sh
python scripts/repro_run.py --mode retrain --key hnp_s0



## Retrained vs released loop, round 3 record (pairs found, in-loop attack success, patch statistics)

In [6]:
# Retrained vs released: the same summary file from both runs (where the retrained run has finished).
import json
rows = {}
released = repro.fetch_results()
for k in KEYS:
    new = repro.RUNS / "retrain" / k / "round3/done.json"
    old = released / Path(reg[k]["run_dir"]).name / "round3/done.json"
    if new.exists() and old.exists():
        rows[(k, "released")] = pd.json_normalize(json.loads(old.read_text())).iloc[0]
        rows[(k, "retrained")] = pd.json_normalize(json.loads(new.read_text())).iloc[0]
pd.DataFrame(rows).T['n_pairs_round', 'n_pairs_buffer', 'round3/train_asr', 'round3/val_asr', 'round3/val_mean_gain'] if rows else "no finished retrained runs yet"

'no finished retrained runs yet'